Install packages

In [ ]:
!pip install diffusers transformers scipy ftfy peft accelerate -q

Generate images with prompts

In [ ]:
from diffusers import DiffusionPipeline
import torch

pipe_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = DiffusionPipeline.from_pretrained(pipe_id, torch_dtype=torch.float16)
pipe.to("cuda")

In [ ]:
prompt = "photo of young woman, sitting outside restaurant, color, wearing dress, " \
         "rim lighting, studio lighting, looking at the camera, up close, perfect eyes"
image = pipe(prompt).images[0]
image

In [ ]:
prompt = "Astronaut in space, realistic, detailed, 8k"
neg_prompt = "ugly, deformed, disfigured, poor details, bad anatomy"
generator = torch.Generator("cuda").manual_seed(127)

image = pipe(prompt, num_inference_steps=50, generator=generator,
             negative_prompt=neg_prompt, height=512, width=912,
             guidance_scale=6
             ).images[0]
image

Loading LoRA Weights

In [ ]:
pipe.load_lora_weights("ostris/ikea-instructions-lora-sdxl",
                       weight_name="ikea_instructions_xl_v1_5.safetensors",
                       adapter_name="ikea",
)

In [ ]:
prompt = "super villan"
image = pipe(prompt, num_inference_steps=30, cross_attention_kwargs={"scale": 0.9},
             generator=torch.manual_seed(125)
            ).images[0]
image

Using ControlNet OpenPose

In [ ]:
!pip install controlnet_aux -q

In [ ]:
from diffusers import ControlNetModel, AutoPipelineForText2Image
from diffusers.utils import load_image
import torch

controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_OpenPose",
                                             torch_dtype=torch.float16, variant="fp16"
                                            ).to("cuda")
original_image = load_image(
    "https://images.pexels.com/photos/1701194/pexels-photo-1701194.jpeg"
)

In [ ]:
from PIL import Image

def image_grid(imgs, rows, cols, resize=256):
    assert len(imgs) == rows * cols
    
    if resize is not None:
        imgs = [img.resize((resize, resize)) for img in imgs]
    w, h = imgs[0].size
    grid_w, grid_h = cols * w, rows * h
    grid = Image.new("RGB", size=(grid_w, grid_h))
    
    for i, img in enumerate(imgs):
        x = i % cols * w
        y = i // cols * h
        grid.paste(img, box=(x, y))
    return grid

In [ ]:
from controlnet_aux import OpenposeDetector

model = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
pose_image = model(original_image)
image_grid([original_image,pose_image], 1, 2)

In [ ]:
controlnet_pipe = AutoPipelineForText2Image.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

In [ ]:
prompt = "a woman dancing in the rain, masterpiece, best quality, enchanting, " \
         "striking, beach background"
neg_prompt = "worst quality, low quality, lowres, monochrome, greyscale, " \
             "multiple views, comic, sketch, bad anatomy, deformed, disfigured, " \
             "watermark, multiple_views, mutation hands, watermark, bad facial"

image = controlnet_pipe(prompt, negative_prompt=neg_prompt, num_images_per_prompt=4,
                        image=pose_image
                        ).images
image_grid(image, 1, 4)